# Feature Engineering — Admisiones de Posgrado

**Autor:** SCO

**Fecha:** 2026-08-19

**Descripción:**
Notebook del Issue #4 "Feature Engineering". Prepara el dataset de admisiones para
modelamiento: elimina duplicados, clasifica las variables según su naturaleza,
define pipelines de imputación, encoding y escalado con `Pipeline` y
`ColumnTransformer` de scikit-learn, y los ajusta únicamente sobre el conjunto de
entrenamiento para evitar data leakage.

No entrena modelos ni crea features artificiales (etapas posteriores del curso).

## 🎯 Alcance y decisiones de preprocessing

Se trabaja sobre el Parquet tipado del Issue #2
(`data/02_intermediate/Admission_Predict.parquet`). Decisiones adoptadas:

1. Eliminar los duplicados **antes** del split train/test.
2. No eliminar ni modificar outliers: documentar y justificar la decisión (ver más abajo).
3. Separar `X` (features) e `y` (target `Chance of Admit`), sin transformar el target.
4. `train_test_split(test_size=0.2, random_state=42)` antes de ajustar transformadores.
5. **Numéricas** (`GRE Score`, `TOEFL Score`, `CGPA`): `SimpleImputer(median)` + `StandardScaler`.
6. **Ordinales** (`University Rating`, `SOP`, `LOR`): `SimpleImputer(most_frequent)` + `OrdinalEncoder` + `StandardScaler`.
7. **Binaria** (`Research`): `SimpleImputer(most_frequent)`, manteniendo 0/1.
8. Todo se combina en un `ColumnTransformer`.
9. Sin data leakage: los transformadores hacen `fit` solo sobre `X_train`.
10. **Feature Engineering**: no se crean features artificiales (combinaciones, `log`,
    `sqrt`, cuadrados, discretización); las hipótesis del EDA (score compuesto,
    interacciones) se evalúan en el modelamiento, no aquí.
11. **Feature Selection**: se conservan las 7 features (todas con asociación
    significativa con el target); la redundancia GRE–TOEFL–CGPA se tratará en
    modelamiento con regularización/selección.
12. No se entrenan modelos.

El diseño sigue el ejemplo del profesor
(<https://joserzapata.github.io/post/ciencia-datos-proyecto-python/4-feat_eng/>),
adaptado a las variables y decisiones de este dataset.

## 📚 Import libraries

In [1]:
# base libraries for data science
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

# Versiones para reproducibilidad
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.13
Pandas: 3.0.5
NumPy: 2.5.2
scikit-learn: 1.9.0


## 💾 Load data

Lectura **reproducible** e **inmutable** del dataset preparado: se localiza la raíz
del repositorio (buscando `pyproject.toml`) y se lee el Parquet tipado producido por
el Issue #2. No se modifica ni el Parquet ni el CSV RAW.

In [2]:
def find_repo_root(start: Path) -> Path:
    """Localiza la raíz del repositorio subiendo hasta encontrar `pyproject.toml`."""
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("No se encontró la raíz del repositorio (pyproject.toml).")


ROOT = find_repo_root(Path.cwd())
DATA_PATH = ROOT / "data" / "02_intermediate" / "Admission_Predict.parquet"

admission_df = pd.read_parquet(DATA_PATH)

print(f"Dataset cargado desde: {DATA_PATH}")
admission_df.head()

Dataset cargado desde: /home/elkiruvi/Proyecto-Admisiones/data/02_intermediate/Admission_Predict.parquet


,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
0,337,118,4,4.5,4.5,9.65,1,0.92
1,324,107,4,4.0,4.5,8.87,1,0.76
2,316,104,3,3.0,3.5,8.00,1,0.72
3,322,110,3,3.5,2.5,8.67,1,0.80
4,314,103,2,2.0,3.0,8.21,0,0.65


## 👷 Preparación de datos: duplicados, valores faltantes y tipos

Antes de definir las transformaciones se caracteriza el dataset: dimensiones,
duplicados, valores faltantes y tipos. La limpieza de duplicados se hace aquí,
**antes** del split, para que la misma fila no aparezca a la vez en train y en test
(fuente de data leakage).

In [3]:
print(f"Dimensiones iniciales (filas, columnas): {admission_df.shape}")
n_duplicated = int(admission_df.duplicated().sum())
print(f"Filas duplicadas (completas): {n_duplicated}")

# Se eliminan los duplicados antes del split train/test.
admission_df = admission_df.drop_duplicates().reset_index(drop=True)

print(f"\nDimensiones tras eliminar duplicados: {admission_df.shape}")
print(f"Registros únicos restantes: {admission_df.shape[0]}")

Dimensiones iniciales (filas, columnas): (623, 8)
Filas duplicadas (completas): 152

Dimensiones tras eliminar duplicados: (471, 8)
Registros únicos restantes: 471


### Valores faltantes y tipos de variables

Se documentan los valores faltantes y el tipo de cada columna tal como vienen del
Parquet del Issue #2. Los nulos se tratarán con imputación dentro de los pipelines
(no se eliminan filas ni columnas por nulos).

In [4]:
print("Valores faltantes por columna:")
print(admission_df.isna().sum().to_string())

print("\nTipos de datos:")
print(admission_df.dtypes.to_string())

Valores faltantes por columna:
GRE Score            11
TOEFL Score          19
University Rating     6
SOP                  16
LOR                   8
CGPA                  2
Research             29
Chance of Admit       0

Tipos de datos:
GRE Score              Int64
TOEFL Score            Int64
University Rating      Int64
SOP                  float64
LOR                  float64
CGPA                 float64
Research               Int64
Chance of Admit      float64


### Naturaleza de las variables

Se listan los valores únicos de las variables candidatas a ordinales y binaria para
justificar su clasificación (orden intrínseco vs. 0/1).

In [5]:
for col in ["University Rating", "SOP", "LOR", "Research"]:
    print(f"{col}: {sorted(admission_df[col].dropna().unique().tolist())}")

University Rating: [1, 2, 3, 4, 5]
SOP: [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
LOR: [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
Research: [0, 1]


## 👨‍🏭 Feature Engineering

Según el EDA del Issue #3, las variables se clasifican por su naturaleza:

- **Numéricas (continuas/discretas):** `GRE Score`, `TOEFL Score`, `CGPA`.
- **Ordinales:** `University Rating` (1–5), `SOP` (1–5) y `LOR` (1–5), que tienen un
  orden intrínseco pero no son magnitudes continuas.
- **Binaria:** `Research` (0/1).

**Feature engineering: por qué no se crean features en este Issue.** El EDA del
Issue #3 dejó identificadas hipótesis que **se conservan para experimentar durante
el modelamiento**, no se descartan:

- Redundancia `GRE`–`TOEFL`–`CGPA` (correlación ≈ 0.84): sugiere evaluar
  regularización o selección de variables.
- *Score académico compuesto* (p. ej. resumir el bloque CGPA+GRE).
- Interacción `University Rating` × `Research`.
- Interacción `CGPA` × `Research`.

Por eso, en este Issue **no** se crean features artificiales ni se aplican
transformaciones (`log`, `sqrt`, cuadrados, discretización). El objetivo aquí es
construir un **preprocessing reproducible** con `Pipeline`/`ColumnTransformer` sin
adelantar la etapa de modelamiento: esas hipótesis se evaluarán en el Issue de
modelamiento, comparando contra un baseline donde sí tiene sentido probar si aportan
mejora. Por el mismo motivo se conservan las **7 features** (todas con asociación
estadísticamente significativa con el target): la selección formal queda fuera de
este Issue.

Tampoco se eliminan ni transforman outliers: los valores extremos detectados en el
EDA (p. ej. `CGPA` = 6.8, `SOP`/`LOR` = 1.0, target = 0.34) están dentro de los
rangos esperados de cada variable y son observaciones reales, no errores; su
tratamiento se re-evaluará, si hace falta, tras comparar el desempeño de los modelos.

In [6]:
target = "Chance of Admit"

numeric_features = ["GRE Score", "TOEFL Score", "CGPA"]
ordinal_features = ["University Rating", "SOP", "LOR"]
binary_features = ["Research"]

all_features = numeric_features + ordinal_features + binary_features

print("Features numéricas:", numeric_features)
print("Features ordinales:", ordinal_features)
print("Feature binaria:  ", binary_features)

Features numéricas: ['GRE Score', 'TOEFL Score', 'CGPA']
Features ordinales: ['University Rating', 'SOP', 'LOR']
Feature binaria:   ['Research']


## 🔀 Separación train/test

Se separa el target de las features y se divide en train (80 %) y test (20 %) con
`random_state=42`. El split se hace **antes** de ajustar cualquier transformador:
todo lo que se aprende de los datos (medianas, modas, categorías, media/desviación)
se calcula únicamente sobre `X_train` y luego se aplica a `X_test`, evitando el
data leakage. El target `Chance of Admit` no se transforma.

In [7]:
X = admission_df[all_features]
y = admission_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")
print(f"\nRango del target (sin transformar): [{y.min():.2f}, {y.max():.2f}]")

X_train: (376, 7), y_train: (376,)
X_test:  (95, 7), y_test:  (95,)

Rango del target (sin transformar): [0.34, 0.97]


## 🧪 Pipelines individuales

Cada grupo de variables tiene su propio `Pipeline`, que encadena imputación y,
cuando corresponde, encoding/escalado:

- **Numéricas:** imputar con la mediana (robusto a outliers) y estandarizar con
  `StandardScaler` (media 0, desviación 1).
- **Ordinales:** imputar con la moda, codificar con `OrdinalEncoder` usando
  categorías ordinales explícitas (`University Rating` 1–5; `SOP` y `LOR` 1–5 en
  pasos de 0.5, en orden creciente) y estandarizar.
- **Binaria:** imputar con la moda y **mantener como 0/1**, sin encoding ni escalado.

In [8]:
numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Categorías ordinales explícitas (escala 1-5) en orden creciente:
# University Rating usa enteros 1-5; SOP y LOR usan pasos de 0.5.
ordinal_categories = [
    [1, 2, 3, 4, 5],
    [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0],
    [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0],
]

ordinal_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(categories=ordinal_categories)),
        ("scaler", StandardScaler()),
    ]
)

binary_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
    ]
)

print("numeric_pipe:", numeric_pipe)
print("\nordinal_pipe:", ordinal_pipe)
print("\nbinary_pipe:", binary_pipe)

numeric_pipe: Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

ordinal_pipe: Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder',
                 OrdinalEncoder(categories=[[1, 2, 3, 4, 5],
                                            [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0,
                                             4.5, 5.0],
                                            [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0,
                                             4.5, 5.0]])),
                ('scaler', StandardScaler())])

binary_pipe: Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent'))])


## 🧩 ColumnTransformer

El `ColumnTransformer` combina los tres pipelines y los asocia a sus columnas. Al
aplicarlo, las columnas se transforman en paralelo y se concatenan en un único
array (o DataFrame) listo para el modelamiento.

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe, numeric_features),
        ("ordinal", ordinal_pipe, ordinal_features),
        ("binary", binary_pipe, binary_features),
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``f

## ⚙️ Ajuste y transformación (fit solo sobre train)

Se ajusta el `ColumnTransformer` **únicamente** sobre `X_train` y luego se
transforman tanto `X_train` como `X_test` con el mismo transformador ya ajustado.
Así, la imputación, el encoding y el escalado usan estadísticas calculadas solo con
los datos de entrenamiento.

In [10]:
preprocessor.fit(X_train)

feature_names = preprocessor.get_feature_names_out()
print("Columnas de salida del preprocesador:")
for name in feature_names:
    print(" ", name)

X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

X_train_processed = pd.DataFrame(X_train_transformed, columns=feature_names)
X_test_processed = pd.DataFrame(X_test_transformed, columns=feature_names)

print(f"\nX_train transformado: {X_train_processed.shape}")
print(f"X_test transformado:  {X_test_processed.shape}")
X_train_processed.head()

Columnas de salida del preprocesador:
  numeric__GRE Score
  numeric__TOEFL Score
  numeric__CGPA
  ordinal__University Rating
  ordinal__SOP
  ordinal__LOR
  binary__Research



X_train transformado: (376, 7)
X_test transformado:  (95, 7)


,numeric__GRE Score,numeric__TOEFL Score,numeric__CGPA,ordinal__University Rating,ordinal__SOP,ordinal__LOR,binary__Research
0,0.396809,0.241653,0.355932,-0.099164,0.107090,0.056195,1.0
1,1.014067,1.391798,1.542521,1.635052,1.598414,1.724286,1.0
2,0.749528,0.734572,0.694957,0.767944,0.604198,0.612225,1.0
3,1.102247,1.063185,1.203495,1.635052,0.604198,1.724286,1.0
4,-1.543145,-0.908492,0.050809,-0.099164,0.604198,0.056195,1.0


### Verificación: ausencia de NaN y consistencia de columnas

Se comprueba que, tras el preprocesamiento, no quedan valores nulos y que las
columnas de train y test son idénticas (mismo orden y nombres).

In [11]:
print("NaN tras el preprocesamiento:")
print("  X_train:", int(X_train_processed.isna().sum().sum()))
print("  X_test: ", int(X_test_processed.isna().sum().sum()))

same_columns = list(X_train_processed.columns) == list(X_test_processed.columns)
print("\nColumnas consistentes entre train y test:", same_columns)
print("Nº de columnas train:", X_train_processed.shape[1])
print("Nº de columnas test: ", X_test_processed.shape[1])

NaN tras el preprocesamiento:
  X_train: 0
  X_test:  0

Columnas consistentes entre train y test: True
Nº de columnas train: 7
Nº de columnas test:  7


In [12]:
print("Tipos de datos tras el preprocesamiento:")
print(X_train_processed.dtypes.to_string())

print("\nEstadísticas (X_train procesado):")
X_train_processed.describe().round(4)

Tipos de datos tras el preprocesamiento:
numeric__GRE Score            float64
numeric__TOEFL Score          float64
numeric__CGPA                 float64
ordinal__University Rating    float64
ordinal__SOP                  float64
ordinal__LOR                  float64
binary__Research              float64

Estadísticas (X_train procesado):


,numeric__GRE Score,numeric__TOEFL Score,numeric__CGPA,ordinal__University Rating,ordinal__SOP,ordinal__LOR,binary__Research
count,376.0000,376.0000,376.0000,376.0000,376.0000,376.0000,376.0000
mean,-0.0000,0.0000,0.0000,-0.0000,-0.0000,0.0000,0.5638
std,1.0013,1.0013,1.0013,1.0013,1.0013,1.0013,0.4966
min,-2.3368,-2.5516,-3.0343,-1.8334,-2.3785,-2.7240,0.0000
25%,-0.7495,-0.7442,-0.7120,-0.9663,-0.8871,-0.4998,0.0000
50%,-0.0441,-0.0870,0.0508,-0.0992,0.1071,0.0562,1.0000
75%,0.7495,0.7346,0.7840,0.7679,0.6042,0.6122,1.0000
max,2.0722,2.0490,2.2545,1.6351,1.5984,1.7243,1.0000


## 📊 Análisis de resultados y conclusiones

- Se eliminaron los duplicados detectados, quedando solo registros únicos; las
  dimensiones exactas (antes/después y de la partición train/test) se muestran en
  las celdas anteriores.
- El preprocesador produce 7 columnas (las mismas 7 features de entrada, sin
  expandir categorías porque no hay variables nominales) y deja el dataset **sin NaN**.
- Las columnas de `X_train` y `X_test` son idénticas, y el transformador se ajustó
  solo sobre train → **sin data leakage**.
- El target `Chance of Admit` se conserva en su escala original (ver rango impreso
  en la celda de separación train/test), listo para el modelamiento.
- Resultado para modelar: numéricas estandarizadas, ordinales codificadas y
  estandarizadas, y `Research` binaria imputada sin escalar.

Estos artefactos quedan listos como input de la etapa de modelamiento (baseline y
selección de modelos), que corresponden a los próximos Issues.

## 💡 Propuestas e ideas (próximos pasos)

- Construir un baseline (por ejemplo, predecir la media del target) y comparar.
- Entrenar y comparar modelos de regresión sobre `X_train_processed`/`y_train`,
  evaluando con RMSE/MAE sobre el test (Issue #5 en adelante).
- Evaluar si el escalado de la binaria o el tratamiento de outliers aportan mejoras
  (solo si el desempeño lo justifica).
- Encapsular el `preprocessor` en un pipeline final junto con el estimador para
  reproducir el flujo completo de inferencia.

## 📖 Referencias

- Ejemplo del profesor (Feature Engineering):
  <https://joserzapata.github.io/post/ciencia-datos-proyecto-python/4-feat_eng/>
- Documentación de scikit-learn: `Pipeline`, `ColumnTransformer`, `SimpleImputer`,
  `OrdinalEncoder`, `StandardScaler`.
- `notebooks/README.md` — convención de nombres de notebooks.
- `data/README.md` — convención de capas de datos.
- `AGENTS.md` — reglas del proyecto (data leakage, pipelines, outliers, etc.).